# SQL Interview Self-Test #1

**Calibration:** Easy → Medium. Targets SQL frequently asked in service-company interviews (TCS / Infosys / Wipro / Capgemini / Accenture tier) and client interviews on real DE projects.

**Format:** Each question has a markdown cell describing the task and a code cell beneath it for your answer. Write your SQL inside the `show("""...""")` call; running the cell prints the result as a DataFrame so you can self-verify.

**Coverage:**
1. Second-highest salary
2. Employees earning more than their manager
3. Top N per group (dense rank)
4. Departments with no employees (anti-join)
5. Customers with orders on 3+ consecutive days
6. Running total (window function)
7. Duplicate emails (GROUP BY + HAVING)
8. Pivot — status counts per month (CASE WHEN aggregation)
9. Average salary per department, including empty depts
10. Monthly revenue with month-over-month growth %

**How to use:** Read the question, write your SQL in the code cell, run it, compare against your mental expected output. When done, ask Claude to grade; canonical answers will be appended to `DE_Interview_Prep.md`.

**Prereqs:** `pip install duckdb pandas` (DuckDB is in-process, no server needed).

## Setup

Run the cell below ONCE. It creates an in-memory DuckDB connection and loads 4 tables: `employees`, `departments`, `customers`, `orders`.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute("""
    CREATE TABLE departments (
        dept_id     INTEGER PRIMARY KEY,
        dept_name   VARCHAR
    );
    INSERT INTO departments VALUES
        (1, 'Engineering'),
        (2, 'Sales'),
        (3, 'Marketing'),
        (4, 'HR'),
        (5, 'Finance');

    CREATE TABLE employees (
        emp_id      INTEGER PRIMARY KEY,
        name        VARCHAR,
        salary      INTEGER,
        dept_id     INTEGER,
        manager_id  INTEGER,
        hire_date   DATE
    );
    INSERT INTO employees VALUES
        (1,  'Anjali',  120000, 1, NULL, '2022-01-15'),
        (2,  'Rohit',    95000, 1, 1,    '2022-03-20'),
        (3,  'Priya',   105000, 1, 1,    '2022-06-10'),
        (4,  'Vikram',  130000, 1, 1,    '2023-01-05'),
        (5,  'Suresh',   85000, 2, 6,    '2022-02-15'),
        (6,  'Meera',    90000, 2, NULL, '2021-11-01'),
        (7,  'Karan',    95000, 2, 6,    '2023-03-15'),
        (8,  'Neha',     70000, 3, NULL, '2022-08-20'),
        (9,  'Arjun',    65000, 3, 8,    '2023-05-10'),
        (10, 'Kavita',   75000, 5, NULL, '2023-09-01');

    CREATE TABLE customers (
        cust_id      INTEGER PRIMARY KEY,
        name         VARCHAR,
        email        VARCHAR,
        signup_date  DATE
    );
    INSERT INTO customers VALUES
        (1, 'Aman',    'aman@gmail.com',    '2025-01-10'),
        (2, 'Rhea',    'rhea@gmail.com',    '2025-01-12'),
        (3, 'Sahil',   'sahil@yahoo.com',   '2025-02-05'),
        (4, 'Pooja',   'pooja@hotmail.com', '2025-02-20'),
        (5, 'Ravi',    'aman@gmail.com',    '2025-03-01'),
        (6, 'Tanvi',   'tanvi@gmail.com',   '2025-03-15'),
        (7, 'Nikhil',  'sahil@yahoo.com',   '2025-04-02'),
        (8, 'Aisha',   'aisha@gmail.com',   '2025-04-20');

    CREATE TABLE orders (
        order_id    INTEGER PRIMARY KEY,
        cust_id     INTEGER,
        order_date  DATE,
        amount      DECIMAL(10,2),
        status      VARCHAR
    );
    INSERT INTO orders VALUES
        (1,  1, '2025-01-15',  5000.00, 'completed'),
        (2,  1, '2025-01-16',  3000.00, 'completed'),
        (3,  1, '2025-01-17',  2000.00, 'cancelled'),
        (4,  2, '2025-01-20',  4500.00, 'completed'),
        (5,  3, '2025-02-01',  6000.00, 'completed'),
        (6,  3, '2025-02-02',  1500.00, 'pending'),
        (7,  4, '2025-02-10',  8000.00, 'completed'),
        (8,  4, '2025-02-11',  2500.00, 'completed'),
        (9,  1, '2025-02-15',  3500.00, 'completed'),
        (10, 5, '2025-03-05', 10000.00, 'completed'),
        (11, 5, '2025-03-06',  4000.00, 'cancelled'),
        (12, 5, '2025-03-07',  2500.00, 'completed'),
        (13, 5, '2025-03-08',  1500.00, 'completed'),
        (14, 6, '2025-03-10',  7000.00, 'pending'),
        (15, 2, '2025-03-15',  5500.00, 'completed'),
        (16, 7, '2025-04-01',  9000.00, 'completed'),
        (17, 7, '2025-04-05',  3000.00, 'cancelled'),
        (18, 8, '2025-04-10', 12000.00, 'completed'),
        (19, 3, '2025-04-15',  4000.00, 'completed'),
        (20, 1, '2025-04-20',  2000.00, 'completed'),
        (21, 4, '2025-05-01',  6500.00, 'completed'),
        (22, 6, '2025-05-03',  8500.00, 'completed'),
        (23, 6, '2025-05-04',  1000.00, 'pending'),
        (24, 2, '2025-05-10',  5000.00, 'completed'),
        (25, 8, '2025-05-15',  7500.00, 'cancelled');
""")

def show(sql):
    """Run a SQL query and return the result as a DataFrame."""
    return con.execute(sql).df()

print('Setup complete. Tables loaded: departments, employees, customers, orders.')

Setup complete. Tables loaded: departments, employees, customers, orders.


## Sample data preview

Run each cell below to see the tables you'll be working with.

In [2]:
show('SELECT * FROM departments')

,dept_id,dept_name
0,1,Engineering
1,2,Sales
2,3,Marketing
3,4,HR
4,5,Finance


In [3]:
show('SELECT * FROM employees ORDER BY dept_id, salary DESC')

,emp_id,name,salary,dept_id,manager_id,hire_date
0,4,Vikram,130000,1,1,2023-01-05
1,1,Anjali,120000,1,<NA>,2022-01-15
2,3,Priya,105000,1,1,2022-06-10
3,2,Rohit,95000,1,1,2022-03-20
4,7,Karan,95000,2,6,2023-03-15
5,6,Meera,90000,2,<NA>,2021-11-01
6,5,Suresh,85000,2,6,2022-02-15
7,8,Neha,70000,3,<NA>,2022-08-20
8,9,Arjun,65000,3,8,2023-05-10
9,10,Kavita,75000,5,<NA>,2023-09-01


In [4]:
show('SELECT * FROM customers')

,cust_id,name,email,signup_date
0,1,Aman,aman@gmail.com,2025-01-10
1,2,Rhea,rhea@gmail.com,2025-01-12
2,3,Sahil,sahil@yahoo.com,2025-02-05
3,4,Pooja,pooja@hotmail.com,2025-02-20
4,5,Ravi,aman@gmail.com,2025-03-01
5,6,Tanvi,tanvi@gmail.com,2025-03-15
6,7,Nikhil,sahil@yahoo.com,2025-04-02
7,8,Aisha,aisha@gmail.com,2025-04-20


In [5]:
show('SELECT * FROM orders ORDER BY order_date')

,order_id,cust_id,order_date,amount,status
0,1,1,2025-01-15,5000.0,completed
1,2,1,2025-01-16,3000.0,completed
2,3,1,2025-01-17,2000.0,cancelled
3,4,2,2025-01-20,4500.0,completed
4,5,3,2025-02-01,6000.0,completed
5,6,3,2025-02-02,1500.0,pending
6,7,4,2025-02-10,8000.0,completed
7,8,4,2025-02-11,2500.0,completed
8,9,1,2025-02-15,3500.0,completed
9,10,5,2025-03-05,10000.0,completed


---
## Q1 — Second-highest salary

Find the **second-highest distinct salary** from the `employees` table.

- If the top two salaries are tied (e.g., both 130000), return the next-highest *distinct* salary below that.
- Return a single row, single column named `second_highest_salary`.

**Tables:** `employees(emp_id, name, salary, dept_id, manager_id, hire_date)`


In [6]:
# Q1: Write your SQL between the triple quotes

show("""
SELECT DISTINCT salary as second_highest_salary
     FROM employees
     ORDER BY salary DESC
     LIMIT 1 OFFSET 1
""")

,second_highest_salary
0,120000


In [7]:
show("""
WITH cte AS (
SELECT salary, dense_rank() over(order by salary desc) as dr
     FROM employees
     )
SELECT salary
     FROM cte
     WHERE dr=2
""")

,salary
0,120000


In [8]:
show("""
SELECT max(salary)
FROM employees
WHERE salary<(SELECT max(salary) FROM employees)
""")

,max(salary)
0,120000


---
## Q2 — Employees earning more than their manager

List every employee whose salary is **greater than their manager's salary**.

- An employee with `manager_id IS NULL` has no manager and should be excluded.
- Return: `employee_name`, `employee_salary`, `manager_name`, `manager_salary`.
- Order by `employee_name` ascending.

**Tables:** `employees(emp_id, name, salary, dept_id, manager_id, hire_date)` — self-referential via `manager_id → emp_id`.

In [9]:
# Q2: Write your SQL between the triple quotes

show("""
SELECT * 
     FROM employees e1
     WHERE e1.salary>(
     SELECT e2.salary
     FROM employees e2
     WHERE e1.manager_id=e2.emp_id
     ORDER BY e1.name
     )
""")

,emp_id,name,salary,dept_id,manager_id,hire_date
0,4,Vikram,130000,1,1,2023-01-05
1,7,Karan,95000,2,6,2023-03-15


In [10]:
show("""
SELECT t1.name as employee_name, t1.salary AS employee_salary, t2.name AS manager_name, t2.salary AS manager_salary
     FROM employees t1 
     JOIN employees t2
     ON t1.manager_id=t2.emp_id
     WHERE t1.salary>t2.salary
     ORDER BY t1.name
""")

,employee_name,employee_salary,manager_name,manager_salary
0,Karan,95000,Meera,90000
1,Vikram,130000,Anjali,120000


---
## Q3 — Top 3 highest-paid employees per department

For each department, return its top 3 highest-paid employees.

- If a department has fewer than 3 employees, return all of them.
- **Ties at the same salary share the same rank** (use `DENSE_RANK` so two employees tied at rank 1 are both included, and rank 2 follows).
- Return: `dept_name`, `emp_name`, `salary`, `rank_in_dept`.
- Order by `dept_name`, then `rank_in_dept`.

**Tables:** `employees`, `departments(dept_id, dept_name)`.

In [11]:
# Q3: Write your SQL between the triple quotes

show("""
WITH cte AS (
     SELECT d.dept_name, e.name as emp_name, e.salary as salary, dense_rank()over(partition by dept_name order by salary DESC) AS rank_in_dept
     FROM employees e
     JOIN departments d
     ON e.dept_id=d.dept_id
     )
SELECT dept_name, emp_name, salary, rank_in_dept
     FROM cte
     WHERE rank_in_dept<=3
     ORDER BY dept_name, rank_in_dept
""")

,dept_name,emp_name,salary,rank_in_dept
0,Engineering,Vikram,130000,1
1,Engineering,Anjali,120000,2
2,Engineering,Priya,105000,3
3,Finance,Kavita,75000,1
4,Marketing,Neha,70000,1
5,Marketing,Arjun,65000,2
6,Sales,Karan,95000,1
7,Sales,Meera,90000,2
8,Sales,Suresh,85000,3


---
## Q4 — Departments with zero employees

Find every department that has **no employees assigned**.

- Return: `dept_id`, `dept_name`.
- Order by `dept_id`.

**Tables:** `departments`, `employees`.


In [12]:
# Q4: Write your SQL between the triple quotes

show("""
WITH cte AS (
    SELECT * FROM employees e FULL OUTER JOIN departments d ON e.dept_id=d.dept_id WHERE emp_id IS NULL 
    )
SELECT dept_id_1, dept_name
     FROM cte
""")

,dept_id_1,dept_name
0,4,HR


FULL OUTER JOIN is overkill — LEFT JOIN from departments is the textbook choice. (Your FULL OUTER + WHERE emp_id IS NULL is logically equivalent to RIGHT JOIN here, which is equivalent to LEFT JOIN from the other side. 

In [13]:
show("""
SELECT d.dept_id, d.dept_name
FROM departments d 
LEFT JOIN employees e
ON d.dept_id=e.dept_id
WHERE e.emp_id IS NULL
ORDER BY d.dept_id
""")

,dept_id,dept_name
0,4,HR


---
## Q5 — Customers with orders on 3+ consecutive calendar days

Find every customer who placed at least one order on **three or more consecutive calendar days**.

- All orders count, regardless of `status`.
- Multiple orders on the same day count as one day (you care about *distinct dates*, not order count).
- Return: `cust_id`.
- Order by `cust_id`.

**Tables:** `orders(order_id, cust_id, order_date, amount, status)`.


In [21]:
# Q5: Write your SQL between the triple quotes

show("""
WITH streaks AS (
SELECT *, CASE WHEN order_date - LAG(order_date, 1)OVER(PARTITION BY cust_id ORDER BY order_date) = 1
THEN 1 ELSE 0 END as streak
FROM orders
),
consecutives AS (
SELECT cust_id, SUM(streak)OVER(PARTITION BY cust_id ORDER BY order_date) as consecutive_days
FROM streaks
)
SELECT cust_id
FROM consecutives
WHERE consecutive_days>=2
""")

,cust_id
0,5
1,5
2,1
3,1
4,1


---
## Q6 — Running total of order amount per customer

For each order, return the **running total of that customer's order amounts** (all statuses), ordered by `order_date` ascending within each customer.

- Return: `cust_id`, `order_date`, `amount`, `running_total`.
- Order by `cust_id`, `order_date`.

**Tables:** `orders`.

In [15]:
# Q6: Write your SQL between the triple quotes

show("""
SELECT cust_id, order_date, amount, SUM(amount)OVER(PARTITION BY cust_id ORDER BY order_date) AS running_total
     FROM orders
     ORDER BY cust_id, order_date
""")

,cust_id,order_date,amount,running_total
0,1,2025-01-15,5000.0,5000.0
1,1,2025-01-16,3000.0,8000.0
2,1,2025-01-17,2000.0,10000.0
3,1,2025-02-15,3500.0,13500.0
4,1,2025-04-20,2000.0,15500.0
5,2,2025-01-20,4500.0,4500.0
6,2,2025-03-15,5500.0,10000.0
7,2,2025-05-10,5000.0,15000.0
8,3,2025-02-01,6000.0,6000.0
9,3,2025-02-02,1500.0,7500.0


---
## Q7 — Duplicate email addresses

Find every email address that appears **more than once** in the `customers` table.

- Return: `email`, `occurrences`.
- Order by `occurrences` descending, then `email` ascending.

**Tables:** `customers(cust_id, name, email, signup_date)`.

In [16]:
# Q7: Write your SQL between the triple quotes

show("""
SELECT email, count(email) AS occurrences
FROM customers 
GROUP BY email
HAVING occurrences>1
ORDER BY occurrences DESC, email
""")

,email,occurrences
0,aman@gmail.com,2
1,sahil@yahoo.com,2


---
## Q8 — Pivot: order status counts per month

For each month in the `orders` table, return **one row** with: month, count of completed orders, count of cancelled orders, count of pending orders.

- Return: `month` (formatted as `YYYY-MM` or as the first day of the month — your choice), `completed_count`, `cancelled_count`, `pending_count`.
- Order by `month`.

**Tables:** `orders`.


Your output: every count = total monthly row count (4, 4, 4 / 5, 5, 5 / etc.). Wrong.

The trap: COUNT() counts non-NULL values. Both 1 and 0 are non-NULL. So COUNT(CASE WHEN status='completed' THEN 1 ELSE 0 END) evaluates to:

match → returns 1 (non-NULL, counted)
no match → returns 0 (non-NULL, also counted)
Result: every row counted regardless of condition. Total per month.

Two fixes — pick one:

Fix A — drop the ELSE 0 so non-matching rows return NULL (excluded by COUNT):


COUNT(CASE WHEN status = 'completed' THEN 1 END) AS completed_count
Fix B — switch to SUM which treats 0 as 0:


SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_count
Both produce the same answer. Fix B (SUM) is more common in interviews — easier to remember the rule (sum of indicators), no NULL gotcha.

The mental model: COUNT is the wrong tool when you have an ELSE clause that produces a number. Either drop the ELSE (let it default to NULL) or use SUM. Drill this — the SAME bug bites everyone the first 3 times they try it.

In [17]:
# Q8: Write your SQL between the triple quotes

show("""
SELECT 
     DATE_TRUNC('month', order_date) AS month,
     COUNT(CASE WHEN status='completed' THEN 1 ELSE 0 END) AS completed_count,
     COUNT(CASE WHEN status='pending' THEN 1 ELSE 0 END) AS pending_count,
     COUNT(CASE WHEN status='cancelled' THEN 1 ELSE 0 END) AS cancelled_count
FROM orders
GROUP BY month
ORDER BY month
""")

,month,completed_count,pending_count,cancelled_count
0,2025-01-01,4,4,4
1,2025-02-01,5,5,5
2,2025-03-01,6,6,6
3,2025-04-01,5,5,5
4,2025-05-01,5,5,5


In [22]:
show("""
SELECT 
     DATE_TRUNC('month', order_date) AS month,
     COUNT(CASE WHEN status='completed' THEN 1 END) AS completed_count,
     COUNT(CASE WHEN status='pending' THEN 1 END) AS pending_count,
     COUNT(CASE WHEN status='cancelled' THEN 1 END) AS cancelled_count
FROM orders
GROUP BY month
ORDER BY month
""")

,month,completed_count,pending_count,cancelled_count
0,2025-01-01,3,0,1
1,2025-02-01,4,1,0
2,2025-03-01,4,1,1
3,2025-04-01,4,0,1
4,2025-05-01,3,1,1


---
## Q9 — Average salary per department (including empty departments)

Return the **average salary per department for every department**, including departments that currently have zero employees.

- Empty departments should show `avg_salary = 0` (not NULL).
- Return: `dept_name`, `avg_salary` (rounded to 2 decimal places).
- Order by `dept_name`.

**Tables:** `departments`, `employees`.

In [18]:
# Q9: Write your SQL between the triple quotes

show("""
WITH cte AS (
SELECT * FROM departments d FULL OUTER JOIN employees e ON d.dept_id=e.dept_id  
    )
SELECT DISTINCT dept_name, COALESCE(ROUND(AVG(salary)OVER(PARTITION BY dept_name), 2), 0) AS avg_salary
FROM cte
ORDER BY dept_name
""")

,dept_name,avg_salary
0,Engineering,112500.0
1,Finance,75000.0
2,HR,0.0
3,Marketing,67500.0
4,Sales,90000.0


In [30]:
show("""
WITH cte AS (
SELECT e.salary as salary, d.dept_name as dept_name FROM departments d LEFT JOIN employees e ON d.dept_id=e.dept_id
)
SELECT dept_name, COALESCE(ROUND(AVG(salary), 2), 0) as avg_salary
FROM cte
GROUP BY dept_name
ORDER BY dept_name
""")

,dept_name,avg_salary
0,Engineering,112500.0
1,Finance,75000.0
2,HR,0.0
3,Marketing,67500.0
4,Sales,90000.0


---
## Q10 — Monthly revenue with month-over-month growth %

From **completed orders only**, return monthly total revenue along with the month-over-month growth percentage.

- Return: `month` (`YYYY-MM` or first-of-month), `revenue`, `mom_growth_pct` (rounded to 2 decimal places).
- The first month's `mom_growth_pct` should be `NULL`.
- Formula: `mom_growth_pct = (current_month_revenue - previous_month_revenue) / previous_month_revenue * 100`.
- Order by `month`.

**Tables:** `orders`.


In [19]:
# Q10: Write your SQL between the triple quotes

show("""
WITH revenues AS (
     SELECT *, SUM(amount)OVER(PARTITION BY DATE_TRUNC('month', order_date) ORDER BY order_date) AS monthly_revenue
     FROM orders
)
SELECT DATE_TRUNC('month', order_date) as month, MAX(monthly_revenue) as monthly_revenue
FROM revenues 
GROUP BY month
ORDER BY month

SELECT 
    DATE_TRUNC('month', order_date) AS month, 
    MAX(monthly_revenue) AS revenue,
    CASE WHEN revenue - LAG(revenue)OVER(PARTITION BY month ORDER BY month) = revenue
    THEN 0 AS mom_growth_pct
""")

ParserException: Parser Error: syntax error at or near "SELECT"

LINE 11: SELECT 
         ^

In [20]:
show("""
WITH revenues AS (
     SELECT DATE_TRUNC('month', order_date) AS month,
     SUM(amount) as revenue
     FROM orders
     WHERE status='completed'
     GROUP BY month
     ORDER BY month
    )
SELECT month, revenue,
     ROUND((revenue-LAG(revenue)OVER(ORDER BY month))/LAG(revenue)OVER(ORDER BY month)*100, 2) AS mom_growth_pct
FROM revenues
ORDER BY month
""")

,month,revenue,mom_growth_pct
0,2025-01-01,12500.0,NaN
1,2025-02-01,20000.0,60.00
2,2025-03-01,19500.0,-2.50
3,2025-04-01,27000.0,38.46
4,2025-05-01,20000.0,-25.93


---
## When you're done

1. Run each cell. Check your output makes sense against the sample data.
2. Paste your 10 answers into chat. Claude will grade each one and append canonical answers to `DE_Interview_Prep.md` under a new SQL section.
3. Time yourself if you want a real-interview signal: easy questions ≤ 3 min each, medium questions ≤ 5 min each. Total target: ≤ 40 min.

**Self-verify quickly before submitting:** does your row count look right? Did you handle NULLs (Q4, Q9)? Did you use DISTINCT where the question asked for distinct (Q1)? Did you order as requested?